# Assignment Beta – Question 2: LoRA vs QLoRA vs DoRA vs AdaLoRA for tool calling
**Syed Ukkashah – 23K-0055**

Base model: **Qwen2.5-1.5B-Instruct**. Data: a 2,000-example slice of **glaive-function-calling-v2** (1,800 train / 200 test).
Hardware: Google Colab **T4 (16 GB)**. The T4 has no fast bf16, so every run uses **fp16** mixed precision.

**How to run:** Runtime → Change runtime type → T4 GPU, then *Run all*.
Every run saves its results to Google Drive as soon as it finishes. If Colab disconnects, just *Run all* again and finished runs are skipped.

## Section 1 – Theory

**(a) Trainable parameters.** LoRA replaces the update $\Delta W$ with $BA$, where $B \in \mathbb{R}^{4096 \times r}$ and $A \in \mathbb{R}^{r \times 4096}$:

$$|A| + |B| = r \cdot 4096 + 4096 \cdot r = 2 \cdot 4096 \cdot 16 = 131{,}072$$

**(b) Percentage of the layer.** $W_0$ has $4096^2 = 16{,}777{,}216$ parameters, so $131{,}072 / 16{,}777{,}216 = 0.78\%$ (exactly $2r/d = 32/4096$).

**(c) Why B = 0 and A random?**
* With $B = 0$, $\Delta W = BA = 0$ at step 0. The adapted model starts out *exactly* equal to the pretrained model, so training begins from known-good behaviour.
* A must be random so that learning can start. The gradient of B is $\partial L/\partial B = (\partial L/\partial h)\,(Ax)^\top$, which is non-zero only because $A \neq 0$. After one step B becomes non-zero, and from then on A receives gradient too.
* **If both were random**, $\Delta W = BA$ would be a random matrix added to every adapted layer *before any learning happens*. The first forward pass would already be a corrupted model, the first loss would be high, and the first gradients would mostly try to undo the noise rather than learn the task. With $\alpha/r$ scaling this can destroy pretrained abilities, here instruction following and JSON formatting.
* **If both were zero**, both gradients would be zero forever. That is why exactly one of them is zero.

**(d) Why does each variant exist?**

* **QLoRA: solves the memory of the frozen base.** Plain LoRA still keeps the full base model in 16-bit, so the frozen weights dominate GPU memory. QLoRA stores the base in **4-bit NormalFloat (NF4)**. NF4 places its 16 levels at the quantiles of a normal distribution, which matches how pretrained weights are distributed, so it loses less information than plain int4. **Double quantisation** also quantises the per-block scale constants. Gradients still flow through the de-quantised weights into 16-bit LoRA adapters. **Paged optimizers** keep optimizer state in unified memory that is paged to CPU RAM when the GPU is about to run out (for example on a long-sequence batch), so a memory spike causes slowdown instead of a crash. Result: a 1.5B–65B model can be fine-tuned on a single GPU.
* **DoRA: solves LoRA's coupled magnitude/direction updates.** DoRA writes each weight column as **magnitude × direction**: $W = m \cdot \dfrac{V}{\|V\|_c}$. The LoRA update is applied only to the direction ($V = W_0 + BA$), and the magnitude vector $m$ is trained separately. Weight-decomposition analysis showed that full fine-tuning changes magnitude and direction somewhat independently, while LoRA tends to change them together. Decoupling them lets DoRA behave more like full fine-tuning, especially at low rank. The cost is that the column norm has to be recomputed during training, which uses extra memory and time.
* **AdaLoRA: solves "same rank everywhere".** Plain LoRA gives every adapted matrix the same rank r, but some layers and matrices matter much more than others. AdaLoRA parametrises each update in SVD form, $P \Lambda Q$, and scores every singular triplet by its **importance**: a smoothed sensitivity $|w \cdot \partial L/\partial w|$ with an uncertainty term. On a schedule it **prunes the least important singular values**, moving a global rank budget toward the matrices that need it, from `init_r` down to an average `target_r`. An orthogonality regulariser keeps P and Q close to orthogonal so that pruning a singular value is clean.

In [ ]:
d, r = 4096, 16
lora_params = d * r + r * d
print(f"(a) trainable params = {d}*{r} + {r}*{d} = {lora_params:,}")
print(f"(b) percentage of W0  = {lora_params:,} / {d*d:,} = {100 * lora_params / (d*d):.4f} %")

## Section 2 – Setup

In [ ]:
# Colab ships an old torchao (0.10) that makes the newest PEFT raise an ImportError; we don't use it, so remove it.
!pip -q uninstall -y torchao
!pip -q install -U transformers peft bitsandbytes accelerate

In [ ]:
import gc, json, math, os, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          Trainer, TrainerCallback, TrainingArguments)
from peft import AdaLoraConfig, LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- one place for every setting that must stay identical across runs ----
MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"
SEED         = 42
N_TOTAL      = 2000
N_TEST       = 200
MAX_LEN      = 768             # examples longer than this are dropped, never truncated
EPOCHS       = 1
LR           = 2e-4
BATCH        = 2
GRAD_ACCUM   = 8               # effective batch = 16
TARGETS      = ["q_proj", "k_proj", "v_proj", "o_proj"]
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
MAX_NEW_TOKENS = 160

SMOKE_TEST = False             # True = tiny run (60 train / 20 test, 10 steps) to check everything works

# results go to Google Drive so a Colab disconnect doesn't lose finished runs
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/AssignmentBeta_Q2")
except ImportError:
    OUT = Path("results")
if SMOKE_TEST:
    OUT = OUT.with_name(OUT.name + "_smoke")    # keep test results away from the real ones
OUT.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Data

A glaive example looks like this:
```
system: SYSTEM: You are a helpful assistant with access to the following functions ... { "name": "get_news", ... }
chat:   USER: ... ASSISTANT: <functioncall> {"name": "get_news", "arguments": '{"category": "tech"}'} <|endoftext|> FUNCTION RESPONSE: ...
```
We keep examples whose **first** assistant turn is a function call. The target is normalised to strict JSON, `{"name": ..., "arguments": {...}}`. Glaive wraps the arguments in single quotes as a string, which `json.loads` would reject, so they are parsed into a real dict.

In [ ]:
INSTRUCTION = ('\n\nWhen a function is needed, reply with ONLY a JSON object of the form '
               '{"name": <function name>, "arguments": {<argument>: <value>}} and nothing else.')


def parse_example(ex):
    # Return {"system", "user", "target", "functions"} or None if the example doesn't fit.
    chat = ex["chat"]
    if "ASSISTANT:" not in chat or "USER:" not in chat:
        return None
    user = chat.split("ASSISTANT:")[0].split("USER:")[-1].strip()
    first_reply = chat.split("ASSISTANT:")[1].split("<|endoftext|>")[0].strip()
    if not first_reply.startswith("<functioncall>"):
        return None
    call = first_reply[len("<functioncall>"):].strip()
    name = re.search(r'"name":\s*"([^"]+)"', call)
    args = re.search(r'"arguments":\s*\'(.*)\'\s*}$', call, re.S)
    if not name or not args:
        return None
    try:
        arguments = json.loads(args.group(1))
    except json.JSONDecodeError:
        return None
    system = ex["system"].replace("SYSTEM:", "", 1).strip()
    return {
        "system": system + INSTRUCTION,
        "user": user,
        "target": json.dumps({"name": name.group(1), "arguments": arguments}),
        "functions": re.findall(r'"name":\s*"([^"]+)"', system),
    }


def prompt_text(ex):
    messages = [{"role": "system", "content": ex["system"]}, {"role": "user", "content": ex["user"]}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# download the raw JSON file directly (avoids the `datasets`/pyarrow version conflicts on Colab)
path = hf_hub_download("glaiveai/glaive-function-calling-v2", "glaive-function-calling-v2.json", repo_type="dataset")
raw = json.load(open(path, encoding="utf-8"))          # list of {"system": ..., "chat": ...}
random.Random(SEED).shuffle(raw)
examples = []
for ex in raw:
    parsed = parse_example(ex)
    if parsed is None:
        continue
    n_tokens = len(tokenizer(prompt_text(parsed) + parsed["target"] + tokenizer.eos_token)["input_ids"])
    if n_tokens <= MAX_LEN:
        examples.append(parsed)
    if len(examples) == N_TOTAL:
        break

train_data, test_data = examples[:N_TOTAL - N_TEST], examples[N_TOTAL - N_TEST:]
if SMOKE_TEST:
    train_data, test_data = train_data[:60], test_data[:20]

# the test set is written to disk once and only touched by evaluate()
(OUT / "test_set.json").write_text(json.dumps(test_data, indent=1))
print(f"train = {len(train_data)}, test = {len(test_data)}")
print("\nExample target:", train_data[0]["target"])

### Tokenisation
The loss is computed **only on the answer tokens**. Prompt tokens get the label `-100`, so the model learns to produce the call, not to repeat the schema.

In [ ]:
def tokenize(ex):
    prompt_ids = tokenizer(prompt_text(ex), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(ex["target"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + answer_ids,
            "labels": [-100] * len(prompt_ids) + answer_ids}


def collate(batch):
    # Right-pad a batch: pad token for inputs, -100 for labels, 0 in the attention mask.
    longest = max(len(x["input_ids"]) for x in batch)
    ids, labels, mask = [], [], []
    for x in batch:
        pad = longest - len(x["input_ids"])
        ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad)
        labels.append(x["labels"] + [-100] * pad)
        mask.append([1] * len(x["input_ids"]) + [0] * pad)
    return {"input_ids": torch.tensor(ids), "labels": torch.tensor(labels), "attention_mask": torch.tensor(mask)}


train_tokens = [tokenize(ex) for ex in train_data]
steps_per_epoch = math.ceil(len(train_tokens) / (BATCH * GRAD_ACCUM))
TOTAL_STEPS = 10 if SMOKE_TEST else steps_per_epoch * EPOCHS
print("optimizer steps:", TOTAL_STEPS)

### Model builder
The **only** thing that changes between runs is the adapter config (and, for QLoRA, the 4-bit base). All runs share the same target modules, alpha, dropout, optimizer (`paged_adamw_32bit`), schedule, seed and data.

In [ ]:
def build_model(method, rank):
    set_seed()
    if method == "qlora":
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map={"": 0})
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map={"": 0})
        if method == "base":
            return model
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()

    common = dict(target_modules=TARGETS, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM")
    if method in ("lora", "qlora"):
        config = LoraConfig(r=rank, **common)
    elif method == "dora":
        config = LoraConfig(r=rank, use_dora=True, **common)
    elif method == "adalora":
        # start at 1.5x the target rank (12 -> 8 for the main run), prune between 10% and 80% of training
        config = AdaLoraConfig(init_r=int(rank * 1.5), target_r=rank, total_step=TOTAL_STEPS,
                               tinit=int(0.1 * TOTAL_STEPS), tfinal=int(0.2 * TOTAL_STEPS), deltaT=5, **common)
    return get_peft_model(model, config)


class AdaLoraCallback(TrainerCallback):
    # AdaLoRA must re-allocate rank after backward but before the optimizer step (gradients still present).
    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        model.base_model.update_and_allocate(state.global_step)

### Evaluation
Greedy decoding on the 200 test prompts, batched with left padding for speed.

In [ ]:
def score(output, ex):
    # Return (json_valid, tool_correct, full_match) for one model output.
    gold = json.loads(ex["target"])
    try:
        pred = json.loads(output.strip())
    except json.JSONDecodeError:
        return False, False, False
    if not isinstance(pred, dict):
        return True, False, False
    tool_ok = pred.get("name") == gold["name"]
    return True, tool_ok, tool_ok and pred.get("arguments") == gold["arguments"]


@torch.no_grad()
def evaluate(model, run_name, batch_size=16):
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    outputs, start, new_tokens = [], time.time(), 0
    for i in range(0, len(test_data), batch_size):
        chunk = test_data[i:i + batch_size]
        enc = tokenizer([prompt_text(ex) for ex in chunk], return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        gen = gen[:, enc["input_ids"].shape[1]:]
        new_tokens += int((gen != tokenizer.pad_token_id).sum())
        outputs += tokenizer.batch_decode(gen, skip_special_tokens=True)
    tokenizer.padding_side = "right"
    seconds = time.time() - start

    rows = []
    for ex, out in zip(test_data, outputs):
        valid, tool_ok, full = score(out, ex)
        rows.append({"user": ex["user"], "functions": ex["functions"], "gold": ex["target"], "output": out,
                     "json_valid": valid, "tool_correct": tool_ok, "full_match": full})
    with open(OUT / f"{run_name}_preds.jsonl", "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    n = len(rows)
    return {"json_valid": sum(r["json_valid"] for r in rows), "json_valid_%": 100 * sum(r["json_valid"] for r in rows) / n,
            "tool_acc_%": 100 * sum(r["tool_correct"] for r in rows) / n,
            "full_match_%": 100 * sum(r["full_match"] for r in rows) / n,
            "eval_tok_per_s": new_tokens / seconds}

### Train + evaluate one configuration

In [ ]:
def run(method, rank):
    name = f"{method}_r{rank}" if method != "base" else "base"
    result_file = OUT / f"{name}.json"
    if result_file.exists():                       # resume support after a Colab disconnect
        print(f"{name}: already done, loading saved result")
        return json.loads(result_file.read_text())

    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = build_model(method, rank)
    result = {"run": name, "method": method, "rank": rank}

    if method == "base":
        trainable, total = 0, sum(p.numel() for p in model.parameters())
        result.update(train_minutes=0.0, peak_mem_GB=torch.cuda.max_memory_allocated() / 1e9)
    else:
        trainable, total = model.get_nb_trainable_parameters()   # counts 4-bit weights correctly
        args = TrainingArguments(
            output_dir=f"/tmp/{name}", num_train_epochs=EPOCHS, max_steps=TOTAL_STEPS,
            per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=max(1, int(0.03 * TOTAL_STEPS)),
            optim="paged_adamw_32bit", fp16=True, seed=SEED, data_seed=SEED,
            logging_steps=10, save_strategy="no", report_to="none", remove_unused_columns=False)
        callbacks = [AdaLoraCallback()] if method == "adalora" else []
        trainer = Trainer(model=model, args=args, train_dataset=train_tokens, data_collator=collate, callbacks=callbacks)

        start = time.time()
        trainer.train()
        result.update(train_minutes=(time.time() - start) / 60,
                      peak_mem_GB=torch.cuda.max_memory_allocated() / 1e9,
                      final_train_loss=trainer.state.log_history[-1].get("train_loss"))
        model.save_pretrained(OUT / f"{name}_adapter")
        del trainer

    result.update({"trainable_params": trainable, "trainable_%": 100 * trainable / total})
    result.update(evaluate(model, name))
    result_file.write_text(json.dumps(result, indent=1))
    print(json.dumps(result, indent=1))
    del model
    gc.collect(); torch.cuda.empty_cache()
    return result

### Run the five configurations

In [ ]:
results = [run("base", 0), run("lora", 8), run("qlora", 8), run("dora", 8), run("adalora", 8)]

table = pd.DataFrame(results)[["run", "trainable_params", "trainable_%", "peak_mem_GB", "train_minutes",
                               "json_valid", "json_valid_%", "tool_acc_%", "full_match_%", "eval_tok_per_s"]]
table.to_csv(OUT / "results_section2.csv", index=False)
table.round(3)

*Note on AdaLoRA's trainable count:* PEFT allocates the full `init_r = 12` and *masks* pruned singular values instead of deleting them. The trainable count therefore reflects r = 12, even though the final **effective** average rank is 8.

## Section 3 – Is a bigger rank actually better?

In [ ]:
methods = table[table.run != "base"].copy()
best = methods.sort_values(["full_match_%", "tool_acc_%"], ascending=False).iloc[0]
best_method = best["run"].split("_")[0]
print("Best method from Section 2:", best_method)

rank_results = [run(best_method, r) for r in (4, 8, 16, 32)]      # r=8 is loaded from Section 2
ranks = pd.DataFrame(rank_results)[["rank", "trainable_params", "peak_mem_GB", "train_minutes",
                                    "json_valid_%", "tool_acc_%", "full_match_%"]]
ranks.to_csv(OUT / "results_section3.csv", index=False)
ranks.round(3)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(ranks["rank"], ranks["tool_acc_%"], "o-", label="tool accuracy")
ax1.plot(ranks["rank"], ranks["full_match_%"], "s--", label="full match")
ax1.set_xscale("log", base=2); ax1.set_xticks(ranks["rank"]); ax1.set_xticklabels(ranks["rank"])
ax1.set_xlabel("LoRA rank r"); ax1.set_ylabel("accuracy on 200 test examples (%)")
ax2 = ax1.twinx()
ax2.bar(ranks["rank"], ranks["peak_mem_GB"], width=ranks["rank"] * 0.25, alpha=0.2, color="gray", label="peak memory")
ax2.set_ylabel("peak GPU memory (GB)")
ax1.legend(loc="lower right"); ax1.set_title(f"{best_method.upper()}: accuracy vs rank")
plt.tight_layout(); plt.savefig(OUT / "rank_vs_accuracy.png", dpi=150); plt.show()

**Conclusion (fill in from the plot):** compare the accuracy gain from r = 8 → 16 → 32 with the change in trainable parameters and peak memory. If the curve is flat within roughly ±1–2 points (2–4 test examples out of 200), the extra rank is not buying anything on this task.

## Section 4 support – failures, merge behaviour

In [ ]:
def classify_error(row):
    out = row["output"].strip()
    gold = json.loads(row["gold"])
    try:
        pred = json.loads(out)
    except json.JSONDecodeError:
        return "refused to call a tool" if "{" not in out else "broken JSON"
    if not isinstance(pred, dict) or "name" not in pred:
        return "broken JSON"
    if pred["name"] not in row["functions"]:
        return "made-up function"
    if pred["name"] != gold["name"]:
        return "wrong tool"
    pred_args = pred.get("arguments") or {}
    if set(pred_args) - set(gold["arguments"]):
        return "made-up argument"
    if set(gold["arguments"]) - set(pred_args):
        return "missing argument"
    return "wrong argument value"


best_run = f"{best_method}_r8"
preds = [json.loads(line) for line in open(OUT / f"{best_run}_preds.jsonl")]
wrong = [p for p in preds if not p["full_match"]]
for p in wrong:
    p["error_type"] = classify_error(p)
print(f"{best_run}: {len(wrong)} / {len(preds)} not fully correct")
print(pd.Series([p["error_type"] for p in wrong]).value_counts())

for p in wrong[:5]:
    print("\n" + "-" * 90)
    print("USER    :", p["user"][:300])
    print("CORRECT :", p["gold"])
    print("MODEL   :", p["output"][:300])
    print("LABEL   :", p["error_type"])

**Merge check (supports the "your call" discussion).** A LoRA or DoRA adapter trained on a 16-bit base can be merged back exactly. A QLoRA adapter was trained against the *4-bit* weights. Merging it into the 16-bit base, which is what deployment usually does, means the adapter now sits on weights it never saw during training. Here we measure what that does to accuracy.

In [ ]:
RUN_MERGE_CHECK = True
if RUN_MERGE_CHECK and not (OUT / "qlora_r8_merged16.json").exists():
    base16 = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map={"": 0})
    merged = PeftModel.from_pretrained(base16, OUT / "qlora_r8_adapter").merge_and_unload()
    merged_result = {"run": "qlora_r8 merged into fp16 base", **evaluate(merged, "qlora_r8_merged16")}
    (OUT / "qlora_r8_merged16.json").write_text(json.dumps(merged_result, indent=1))
    del base16, merged; gc.collect(); torch.cuda.empty_cache()

if (OUT / "qlora_r8_merged16.json").exists():
    merged_result = json.loads((OUT / "qlora_r8_merged16.json").read_text())
    qlora = json.loads((OUT / "qlora_r8.json").read_text())
    print(pd.DataFrame([{"run": "qlora_r8 (4-bit base, unmerged)", **{k: qlora[k] for k in merged_result if k != "run"}},
                        merged_result]).round(2))

## Files produced (in Drive → `AssignmentBeta_Q2/`)
* `results_section2.csv`, `results_section3.csv`: tables for the report
* `rank_vs_accuracy.png`: the Section 3 plot
* `*_preds.jsonl`: every test output, used for the failure analysis
* `*_adapter/`: the trained adapters